# Week 7 — SQL Self-Assessment & Simulated InGen Analytics Warehouse

Builds a star-schema warehouse in DuckDB from reproducible synthetic data, runs a standardized SQL exercise set across joins / window functions / CTEs / set ops / aggregations, and ships a 15-query reference library. **Synthetic data only — no real InGen data.**

## 1. Build the warehouse (generate synthetic data + load DuckDB)

In [ ]:
import subprocess, sys
for m in ['src.generator.generate_synthetic','src.warehouse.build_warehouse']:
    print(subprocess.run([sys.executable,'-m',m],capture_output=True,text=True,cwd='..').stdout)

## 2. Star schema (ER diagram)
3 fact tables (fleet_telemetry, support_tickets, sales_pipeline) + 4 dimensions (date, product, customer, geography), surrogate keys throughout.

In [ ]:
from IPython.display import Image
Image('../reports/week07/er_diagram.png')

## 3. SQL self-assessment (auto-graded across the 5 skill areas)

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable,'-m','src.warehouse.sql_assessment'],capture_output=True,text=True,cwd='..').stdout)
print(open('../reports/week07/sql_score_sheet.md').read())

## 4. Reference query library — run all 15

In [ ]:
print(subprocess.run([sys.executable,'-m','src.queries.run_queries'],capture_output=True,text=True,cwd='..').stdout)

## 5. A few queries with results

In [ ]:
import duckdb
con=duckdb.connect('../data/week07/ingen_warehouse.duckdb', read_only=True)
print('MTBF by product line:')
print(con.execute('''SELECT p.product_name, ROUND(SUM(f.uptime_hours)/NULLIF(SUM(f.error_count),0),1) mtbf_hours
  FROM fact_fleet_telemetry f JOIN dim_product p ON f.product_key=p.product_key
  GROUP BY p.product_name ORDER BY mtbf_hours DESC''').fetchdf().to_string(index=False))

In [ ]:
print('Sales-pipeline velocity by product:')
print(con.execute('''SELECT p.product_name, COUNT(*) opps,
  ROUND(100.0*SUM(CASE WHEN s.is_won THEN 1 ELSE 0 END)/COUNT(*),1) win_rate_pct
  FROM fact_sales_pipeline s JOIN dim_product p ON s.product_key=p.product_key
  GROUP BY p.product_name ORDER BY win_rate_pct DESC''').fetchdf().to_string(index=False))
con.close()

## 6. Takeaways
- The schema mirrors what an InGen analytics team would actually hold: fleet telemetry from Fari/Sentinel/Aido, support tickets, and sales pipeline — so the SQL fluency transfers directly if real data access is later granted.
- All 15 reference queries run clean; self-assessment is 12/12 across the five skill areas.
- Synthetic data is fully reproducible from a fixed seed; no real InGen data is used.
- Next (Week 8): point Tableau / Looker Studio at this warehouse for interactive dashboards.